In [2]:
import os
import shutil
import rasterio
import numpy as np
from rasterio.enums import Resampling

# Define the paths for the Sentinel-2 and GT dataset folders
sentinel2_path = 'D:/Kansas/data/10_counties_10km/satellite_sampled/'
gt_path = 'D:/Kansas/data/10_counties_10km/gt_png/'
output_sentinel2_path = 'D:/Kansas/data/10_counties_10km/matched_shape/satellite'
output_gt_path = 'D:/Kansas/data/10_counties_10km/matched_shape/gt'

# Ensure the output paths exist
os.makedirs(output_sentinel2_path, exist_ok=True)
os.makedirs(output_gt_path, exist_ok=True)

def crop_image(image_path, crop_size=1):
    with rasterio.open(image_path) as src:
        data = src.read()
        cropped_data = data[:, crop_size:-crop_size, crop_size:-crop_size]
        profile = src.profile
        profile.update(height=cropped_data.shape[1], width=cropped_data.shape[2])
        return cropped_data, profile

def crop_center(image_path, target_shape):
    with rasterio.open(image_path) as src:
        data = src.read()
        h, w = target_shape
        center_y, center_x = data.shape[1] // 2, data.shape[2] // 2
        start_y, start_x = center_y - h // 2, center_x - w // 2
        cropped_data = data[:, start_y:start_y + h, start_x:start_x + w]
        profile = src.profile
        profile.update(height=h, width=w)
        return cropped_data, profile

# Process GT images
for county_file in os.listdir(gt_path):
    county_name = county_file.replace('.png', '')
    gt_image_path = os.path.join(gt_path, county_file)
    gt_data, gt_profile = crop_image(gt_image_path)
    output_gt_image_path = os.path.join(output_gt_path, county_file)
    with rasterio.open(output_gt_image_path, 'w', **gt_profile) as dst:
        dst.write(gt_data)

    # Process corresponding Sentinel-2 images
    sentinel2_county_path = os.path.join(sentinel2_path, county_name)
    output_sentinel2_county_path = os.path.join(output_sentinel2_path, county_name)
    os.makedirs(output_sentinel2_county_path, exist_ok=True)

    if os.path.exists(sentinel2_county_path):
        for img in os.listdir(sentinel2_county_path):
            img_path = os.path.join(sentinel2_county_path, img)
            target_shape = (gt_data.shape[1] * 3, gt_data.shape[2] * 3)
            cropped_data, cropped_profile = crop_center(img_path, target_shape)
            output_sentinel2_img_path = os.path.join(output_sentinel2_county_path, img)
            with rasterio.open(output_sentinel2_img_path, 'w', **cropped_profile) as dst:
                dst.write(cropped_data)

print("Preprocessing completed successfully!")

Preprocessing completed successfully!
